# Approach 4 (v2): sentence embeddings + linear SVM for ticket category / subcategory

**Goal.** Suggest the category and subcategory of a new support ticket from the text the client writes
(short description + description), so it can be routed without manual triage.

**Models compared**
| Encoder | Classifier | Notes |
|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` (frozen, 384-d, normalised) | `LinearSVC` | the approach under study |
| TF-IDF (word 1–2-grams + character 3–5-grams) | `LinearSVC` | lexical baseline, always run |
| Template lookup (short description → most common subcategory in training) | – | memorisation benchmark |

**Design**
* Target is the **subcategory** (59 classes). Each subcategory has exactly one parent category, so the
  category (17 classes) is read off the hierarchy and can never contradict the subcategory.
* `C` and `class_weight` are chosen by **validation macro-F1**. Test data is never used for selection.
* Confidence comes from sigmoid calibration of the SVM. Tickets below a confidence threshold (chosen on
  held-out-template predictions for ≥ 98 % accuracy on auto-routed tickets) are sent to a human.
* **Unseen-wording stress test.** The data is heavily templated: every short-description template in
  validation and test already occurs in training, so the chronological test mostly measures memorisation.
  Grouped cross-validation that holds out whole templates measures how the model copes with new phrasing.
  This is also the test that decides which encoder is deployed.

> Data are synthetic. Results are a prototype evaluation, not evidence of live routing accuracy.

In [ ]:
import json, os, platform, sys, warnings
from importlib import metadata
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore', category=FutureWarning)
BASE = Path.cwd()
if not (BASE / 'data' / 'train.csv').exists():
    raise RuntimeError('Run this notebook from the repo root (the folder that contains data/). '
                       'If data/ is empty, run `python prepare_data.py` first.')
sys.path.insert(0, str(BASE))
from text_cleaning import model_text as build_model_text  # noqa: E402
from model_utils import TemperatureScaledSVM, fit_temperature  # noqa: E402

DATA, RESULTS, ENCODER_DIR = BASE / 'data', BASE / 'results', BASE / 'encoder'
RESULTS.mkdir(exist_ok=True)
SEED = 42
np.random.seed(SEED)
ENCODER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]
CLASS_WEIGHTS = [None, 'balanced']
TARGET_AUTO_ACCURACY = 0.98      # accuracy required on tickets the model routes on its own
N_FOLDS = 5

## 1. Data

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
val = pd.read_csv(DATA / 'validation.csv')
test = pd.read_csv(DATA / 'test.csv')
unlabelled = pd.read_csv(DATA / 'unlabelled.csv')
hierarchy = pd.read_csv(DATA / 'label_hierarchy.csv')
SUB_TO_CAT = dict(zip(hierarchy.subcategory, hierarchy.category))
for d in (train, val, test, unlabelled):
    d['model_text'] = d['model_text'].fillna('')
    d['short_text'] = d['short_text'].fillna('')
    d['body_text'] = d['body_text'].fillna('')

print(json.dumps(json.loads((DATA / 'preparation_report.json').read_text())['partitions'], indent=1))
print(f'unlabelled tickets to route: {len(unlabelled)}')
print(f'{len(hierarchy)} subcategories under {hierarchy.category.nunique()} categories')
train.sample(5, random_state=SEED)[['model_text', 'category', 'subcategory']]

## 2. Encoders
The sentence encoder is frozen: each unique text is encoded once and cached. If `sentence-transformers` or
the model download is unavailable, the MiniLM rows are **skipped** (not replaced by a stand-in), and the
manifest records this. Only the TF-IDF baseline then runs.

In [ ]:
def load_sentence_encoder():
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as e:  # package missing
        return None, f'sentence-transformers not importable ({type(e).__name__}: {e})'
    try:
        if (ENCODER_DIR / 'config.json').exists() or (ENCODER_DIR / 'modules.json').exists():
            return SentenceTransformer(str(ENCODER_DIR), device='cpu'), f'loaded local copy from {ENCODER_DIR}'
        enc = SentenceTransformer(ENCODER_NAME, device='cpu')
        enc.save(str(ENCODER_DIR))
        return enc, f'downloaded {ENCODER_NAME} and saved to {ENCODER_DIR}'
    except Exception as e:  # download blocked etc.
        return None, f'could not load {ENCODER_NAME} ({type(e).__name__}: {e})'


sentence_encoder, encoder_message = load_sentence_encoder()
HAS_MINILM = sentence_encoder is not None
print('MiniLM:', 'AVAILABLE' if HAS_MINILM else 'SKIPPED', '-', encoder_message)

_emb_cache = {}
def embed(texts):
    """Normalised MiniLM embeddings, cached by text."""
    texts = list(texts)
    new = sorted({t for t in texts if t not in _emb_cache})
    if new:
        vecs = sentence_encoder.encode(new, batch_size=64, normalize_embeddings=True, show_progress_bar=len(new) > 500)
        _emb_cache.update(zip(new, vecs))
    return np.vstack([_emb_cache[t] for t in texts])


class TfidfEncoder:
    """Word 1-2-grams + character 3-5-grams, fitted on the training texts only."""
    def fit(self, texts):
        self.vec = FeatureUnion([
            ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)),
            ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        ]).fit(list(texts))
        return self
    def transform(self, texts):
        return self.vec.transform(list(texts))


class MiniLMEncoder:
    def fit(self, texts):
        return self
    def transform(self, texts):
        return embed(texts)


ENCODERS = {'tfidf': TfidfEncoder}
if HAS_MINILM:
    ENCODERS = {'minilm': MiniLMEncoder, **ENCODERS}
    _ = embed(pd.concat([train, val, test, unlabelled]).model_text)   # encode everything once
    print(f'cached {len(_emb_cache)} unique texts')

## 3. Tune `C` and `class_weight` on validation (subcategory macro-F1)

In [ ]:
def svm(C, cw):
    return LinearSVC(C=C, class_weight=cw, random_state=SEED, max_iter=20000)

def dedup(texts, labels):
    """Collapse identical (text, label) rows into one row weighted by its count. The masked text is highly
    repetitive (about 1k unique texts per 20k tickets), so this gives the same model ~20x faster."""
    g = pd.DataFrame({'t': list(texts), 'y': list(labels)}).groupby(['t', 'y']).size().reset_index(name='w')
    return g.t, g.y.values, g.w.values.astype(float)

def fit_weighted(est, enc, texts, labels):
    t, y, w = dedup(texts, labels)
    return est.fit(enc.transform(t), y, sample_weight=w)

rows, X = [], {}
for enc_name, Enc in ENCODERS.items():
    enc = Enc().fit(train.model_text)
    X[enc_name] = {'enc': enc, 'val': enc.transform(val.model_text)}
    for C in C_GRID:
        for cw in CLASS_WEIGHTS:
            pred = fit_weighted(svm(C, cw), enc, train.model_text, train.subcategory).predict(X[enc_name]['val'])
            rows.append({'encoder': enc_name, 'C': C, 'class_weight': str(cw),
                         'val_subcat_accuracy': accuracy_score(val.subcategory, pred),
                         'val_subcat_macro_f1': f1_score(val.subcategory, pred, average='macro'),
                         'val_category_accuracy': accuracy_score(val.category, pd.Series(pred).map(SUB_TO_CAT))})
selection = pd.DataFrame(rows).sort_values(['encoder', 'val_subcat_macro_f1', 'C'], ascending=[True, False, True])
selection.to_csv(RESULTS / 'validation_model_selection.csv', index=False)
best = {e: g.iloc[0] for e, g in selection.groupby('encoder')}   # tie -> smaller C (more regularised)
BEST = {e: {'C': float(r.C), 'class_weight': None if r.class_weight == 'None' else r.class_weight} for e, r in best.items()}
print('selected:', BEST)
selection.groupby('encoder').head(4).round(4)

## 4. Confidence calibration and the auto-route threshold
Validation cannot set the threshold: every validation template was seen in training, so validation accuracy
is ~100 % at any threshold. Instead both the calibration and the threshold use **held-out-template**
predictions: 5-fold `GroupKFold` on the training set, full text, grouped by short-description template, so
every scored ticket's short description was never seen by its model.
* Calibration: a single temperature `T` so that `softmax(SVM margin / T)` has the lowest log-loss on those
  out-of-fold margins (`model_utils.py`).
* Threshold: the lowest top-1 probability at which the out-of-fold predictions are ≥ the target accuracy, so
  the model only auto-routes when it is confident even on unfamiliar wording.

In [ ]:
def choose_threshold(proba, classes, y_true, target=TARGET_AUTO_ACCURACY):
    top = proba.max(1); pred = classes[proba.argmax(1)]; ok = pred == np.asarray(y_true)
    curve = []
    for t in np.round(np.arange(0.0, 1.0, 0.01), 2):
        m = top >= t
        curve.append({'threshold': t, 'coverage': m.mean(), 'auto_accuracy': ok[m].mean() if m.any() else np.nan})
    curve = pd.DataFrame(curve)
    good = curve[curve.auto_accuracy >= target]
    return (float(good.threshold.min()) if len(good) else 1.0), curve

def fit_model(enc_name, enc, texts, labels, temperature):
    p = BEST[enc_name]
    return TemperatureScaledSVM(fit_weighted(svm(p['C'], p['class_weight']), enc, texts, labels), temperature)

CLASSES = np.array(sorted(train.subcategory.unique()))
y_idx = np.searchsorted(CLASSES, train.subcategory.values)
models, thresholds, temperatures, curves, thr_rows = {}, {}, {}, [], []
thr_folds = list(GroupKFold(n_splits=N_FOLDS).split(train, groups=train.short_template))
for enc_name, Enc in ENCODERS.items():
    p = BEST[enc_name]
    margins = np.full((len(train), len(CLASSES)), -10.0)     # class absent from a fold's training -> very unlikely
    for tr, te in thr_folds:
        enc = Enc().fit(train.model_text.iloc[tr])
        m = fit_weighted(svm(p['C'], p['class_weight']), enc, train.model_text.iloc[tr], train.subcategory.iloc[tr])
        margins[np.ix_(te, np.searchsorted(CLASSES, m.classes_))] = m.decision_function(enc.transform(train.model_text.iloc[te]))
    T = fit_temperature(margins, y_idx)
    oof = np.exp((margins - margins.max(1, keepdims=True)) / T); oof /= oof.sum(1, keepdims=True)
    t, curve = choose_threshold(oof, CLASSES, train.subcategory)
    temperatures[enc_name], thresholds[enc_name] = T, t
    curves.append(curve.assign(encoder=enc_name, split='heldout_template_cv'))
    pred = CLASSES[oof.argmax(1)]; mk = oof.max(1) >= t
    thr_rows.append({'encoder': enc_name, 'temperature': T, 'threshold': t,
                     'oof_accuracy_all': (pred == train.subcategory).mean(),
                     'oof_coverage_at_threshold': mk.mean(),
                     'oof_accuracy_at_threshold': (pred[mk] == train.subcategory.values[mk]).mean() if mk.any() else np.nan})
    models[enc_name] = fit_model(enc_name, X[enc_name]['enc'], train.model_text, train.subcategory, T)
threshold_table = pd.DataFrame(thr_rows)
threshold_table.to_csv(RESULTS / 'auto_route_threshold.csv', index=False)
threshold_table.round(4)

## 5. Chronological test set (Jun 2024, never used for selection)

In [ ]:
def topk_acc(proba, classes, y, k=3):
    topk = classes[np.argsort(-proba, 1)[:, :k]]
    return float(np.mean([yi in row for yi, row in zip(y, topk)]))

def score(y_sub, pred_sub, name, split, proba=None, classes=None, threshold=None, n=None):
    y_cat, p_cat = pd.Series(y_sub).map(SUB_TO_CAT), pd.Series(pred_sub).map(SUB_TO_CAT)
    r = {'model': name, 'split': split, 'n': len(y_sub),
         'subcat_accuracy': accuracy_score(y_sub, pred_sub),
         'subcat_macro_f1': f1_score(y_sub, pred_sub, average='macro'),
         'category_accuracy': accuracy_score(y_cat, p_cat),
         'category_macro_f1': f1_score(y_cat, p_cat, average='macro')}
    if proba is not None:
        r['subcat_top3_accuracy'] = topk_acc(proba, classes, y_sub)
        top = proba.max(1); m = top >= threshold
        r.update({'threshold': threshold, 'auto_route_coverage': m.mean(),
                  'auto_route_accuracy': accuracy_score(np.asarray(y_sub)[m], np.asarray(pred_sub)[m]) if m.any() else np.nan})
    return r

# Benchmarks
majority = train.subcategory.mode()[0]
lookup = train.assign(k=train.short_text.str.lower()).groupby('k').subcategory.agg(lambda s: s.mode()[0])
lookup_pred = test.short_text.str.lower().map(lookup).fillna(majority)

metrics, test_out = [], test[['ticket_id', 'created_at', 'model_text', 'is_service_form', 'is_japanese',
                              'category', 'subcategory']].copy()
metrics.append(score(test.subcategory, np.repeat(majority, len(test)), 'majority_class', 'test'))
metrics.append(score(test.subcategory, lookup_pred.values, 'template_lookup', 'test'))
for enc_name, m in models.items():
    Xt = X[enc_name]['enc'].transform(test.model_text)
    proba = m.predict_proba(Xt); pred = m.classes_[proba.argmax(1)]
    metrics.append(score(test.subcategory, pred, f'{enc_name}_svm', 'test', proba, m.classes_, thresholds[enc_name]))
    test_out[f'{enc_name}_pred_subcategory'] = pred
    test_out[f'{enc_name}_pred_category'] = pd.Series(pred).map(SUB_TO_CAT).values
    test_out[f'{enc_name}_confidence'] = proba.max(1)
    t, curve = choose_threshold(proba, m.classes_, test.subcategory)
    curves.append(curve.assign(encoder=enc_name, split='test'))
    for sl, mask in [('service_form', test.is_service_form), ('japanese', test.is_japanese),
                     ('email', ~test.is_service_form & ~test.is_japanese)]:
        if mask.any():
            metrics.append(score(test.subcategory[mask], pred[mask.values], f'{enc_name}_svm', f'test[{sl}]'))
    # flat category model (category predicted directly) for comparison with the hierarchy read-off
    p = BEST[enc_name]
    flat = fit_weighted(svm(p['C'], p['class_weight']), X[enc_name]['enc'], train.model_text, train.category).predict(Xt)
    metrics.append({'model': f'{enc_name}_svm_flat_category', 'split': 'test', 'n': len(test),
                    'category_accuracy': accuracy_score(test.category, flat),
                    'category_macro_f1': f1_score(test.category, flat, average='macro')})

test_metrics = pd.DataFrame(metrics)
test_metrics.to_csv(RESULTS / 'test_metrics.csv', index=False)
test_out.to_csv(RESULTS / 'test_predictions.csv', index=False)
pd.concat(curves).to_csv(RESULTS / 'auto_route_curve.csv', index=False)
test_metrics.round(4)

## 6. Unseen-wording stress test (grouped cross-validation on train + validation)
Two complementary tests, each 5-fold `GroupKFold`, so a template is never in both the training and the
scoring fold. The test set is not used here, so this result can be used to choose the deployed encoder.
* **New short description:** input = short description only; groups = short-description template.
* **New description body:** input = description body only; groups = body template.
The template lookup scores ~0 here by construction (the held-out template was never seen), which shows
how much of the chronological score is memorisation.

In [ ]:
dev = pd.concat([train, val], ignore_index=True)
cv_rows, cv_oof = [], {}
for test_name, text_col, group_col in [('new_short_description', 'short_text', 'short_template'),
                                       ('new_description_body', 'body_text', 'body_template')]:
    d = dev[dev[text_col].str.len() > 0].reset_index(drop=True)
    folds = list(GroupKFold(n_splits=N_FOLDS).split(d, groups=d[group_col]))
    for enc_name, Enc in ENCODERS.items():
        p = BEST[enc_name]; oof = np.empty(len(d), dtype=object)
        for tr, te in folds:
            enc = Enc().fit(d[text_col].iloc[tr])
            oof[te] = fit_weighted(svm(p['C'], p['class_weight']), enc, d[text_col].iloc[tr], d.subcategory.iloc[tr]) \
                      .predict(enc.transform(d[text_col].iloc[te]))
        cv_oof[(test_name, enc_name)] = oof
        cv_rows.append(score(d.subcategory.values, oof, f'{enc_name}_svm', test_name) | {'templates': d[group_col].nunique()})
    lk = np.empty(len(d), dtype=object)
    for tr, te in folds:
        table = d.iloc[tr].assign(k=d[text_col].iloc[tr].str.lower()).groupby('k').subcategory.agg(lambda s: s.mode()[0])
        lk[te] = d[text_col].iloc[te].str.lower().map(table).fillna(d.subcategory.iloc[tr].mode()[0]).values
    cv_rows.append(score(d.subcategory.values, lk, 'template_lookup', test_name) | {'templates': d[group_col].nunique()})

unseen = pd.DataFrame(cv_rows)
unseen.to_csv(RESULTS / 'unseen_wording_cv.csv', index=False)
unseen.round(4)

## 7. Choose the deployed encoder and fit it on all labelled tickets
Rule: higher mean subcategory macro-F1 across the two unseen-wording tests (in-distribution validation is
saturated, so it cannot separate the encoders). The deployed model is refitted on train + validation + test
with the selected `C`, `class_weight` and threshold; the test metrics above come from the train-only model.

In [ ]:
gen = unseen[unseen.model.str.endswith('_svm')].groupby('model').subcat_macro_f1.mean()
DEPLOY = gen.idxmax().replace('_svm', '')
print('unseen-wording macro-F1:', gen.round(4).to_dict(), '-> deploy:', DEPLOY)

all_lab = pd.concat([train, val, test], ignore_index=True)
final_enc = ENCODERS[DEPLOY]().fit(all_lab.model_text)
final_model = fit_model(DEPLOY, final_enc, all_lab.model_text, all_lab.subcategory, temperatures[DEPLOY])

bundle = {'encoder': DEPLOY, 'encoder_name': ENCODER_NAME if DEPLOY == 'minilm' else 'tfidf word1-2 + char3-5',
          'vectorizer': final_enc.vec if DEPLOY == 'tfidf' else None,
          'classifier': final_model, 'threshold': thresholds[DEPLOY], 'temperature': temperatures[DEPLOY],
          'sub_to_category': SUB_TO_CAT, 'hyperparameters': BEST[DEPLOY],
          'note': 'Input must be cleaned with text_cleaning.model_text(short_description, description).'}
joblib.dump(bundle, RESULTS / 'category_model_svm.joblib')

## 8. Suggest categories for the unlabelled tickets
These 600 tickets were routed to Service Desk L1/L2 without a category — the real use-case for the model.
`auto_route` is True when the top suggestion clears the threshold; the rest go to a human with the top-3
suggestions attached.

In [ ]:
proba = final_model.predict_proba(final_enc.transform(unlabelled.model_text))
order = np.argsort(-proba, 1)[:, :3]; cls = final_model.classes_
sug = unlabelled[['ticket_id', 'created_at', 'ASSIGNMENT_GROUP', 'model_text']].copy()
for k in range(3):
    sug[f'suggestion_{k+1}_subcategory'] = cls[order[:, k]]
    sug[f'suggestion_{k+1}_category'] = pd.Series(cls[order[:, k]]).map(SUB_TO_CAT).values
    sug[f'suggestion_{k+1}_probability'] = proba[np.arange(len(proba)), order[:, k]].round(4)
sug['auto_route'] = sug.suggestion_1_probability >= thresholds[DEPLOY]
sug.to_csv(RESULTS / 'unlabelled_predictions.csv', index=False)
print(f"auto-routed: {sug.auto_route.sum()} of {len(sug)} ({sug.auto_route.mean():.1%}); "
      f"the rest go to a human with top-3 suggestions")
sug.suggestion_1_category.value_counts().head(17)

## 9. Error analysis and charts

In [ ]:
pred_col = f'{DEPLOY}_pred_subcategory'
rep = pd.DataFrame(classification_report(test.subcategory, test_out[pred_col], output_dict=True, zero_division=0)).T
rep = rep.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
rep.insert(0, 'category', rep.index.map(SUB_TO_CAT))
rep.sort_values('f1-score').to_csv(RESULTS / 'test_per_subcategory.csv')

confusions = (test_out[test_out.subcategory != test_out[pred_col]]
              .groupby(['subcategory', pred_col]).size().rename('count').reset_index()
              .sort_values('count', ascending=False))
confusions.to_csv(RESULTS / 'test_top_confusions.csv', index=False)

# unseen-wording confusions show which subcategories are genuinely close in meaning
d = dev[dev.short_text.str.len() > 0].reset_index(drop=True)
oof = cv_oof[('new_short_description', DEPLOY)]
unseen_conf = (pd.DataFrame({'true': d.subcategory, 'pred': oof}).query('true != pred')
               .groupby(['true', 'pred']).size().rename('count').reset_index().sort_values('count', ascending=False))
unseen_conf.to_csv(RESULTS / 'unseen_wording_top_confusions.csv', index=False)
print('Hardest subcategories when the short description wording is new:')
print(pd.DataFrame(classification_report(d.subcategory, oof, output_dict=True, zero_division=0)).T
      .drop(index=['accuracy', 'macro avg', 'weighted avg']).sort_values('f1-score').head(8)[['precision', 'recall', 'f1-score']].round(3))
unseen_conf.head(10)

In [ ]:
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot = pd.concat([test_metrics[test_metrics.split == 'test'].assign(setting='Chronological test (seen templates)'),
                  unseen.assign(setting=unseen.split.map({'new_short_description': 'Unseen short description',
                                                          'new_description_body': 'Unseen description body'}))])
plot = plot[plot.model.isin(['template_lookup'] + [f'{e}_svm' for e in ENCODERS])]
piv = plot.pivot_table(index='setting', columns='model', values='subcat_macro_f1')
piv = piv.loc[[s for s in ['Chronological test (seen templates)', 'Unseen short description', 'Unseen description body'] if s in piv.index]]
piv.plot.bar(ax=axes[0], rot=0, width=0.75, color=['#4C72B0', '#DD8452', '#8C8C8C'][:piv.shape[1]])
axes[0].set_ylabel('Subcategory macro-F1'); axes[0].set_ylim(0, 1.05); axes[0].set_xlabel('')
axes[0].set_title('Seen vs unseen wording'); axes[0].legend(frameon=False, fontsize=8)
axes[0].tick_params(axis='x', labelsize=8)
cv = pd.concat(curves); cv = cv[(cv.split == 'heldout_template_cv')]
for e, g in cv.groupby('encoder'):
    axes[1].plot(g.coverage, g.auto_accuracy, label=e)
    t = thresholds[e]; row = g.loc[(g.threshold - t).abs().idxmin()]
    axes[1].scatter([row.coverage], [row.auto_accuracy], s=25)
axes[1].set_xlabel('Share of tickets auto-routed (held-out-template CV)'); axes[1].set_ylabel('Accuracy on auto-routed')
axes[1].set_title('Coverage vs accuracy (dot = chosen threshold)'); axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(RESULTS / 'evaluation_charts.png', bbox_inches='tight'); plt.show()

cats = sorted(hierarchy.category.unique())
cm = confusion_matrix(test.category, test_out[f'{DEPLOY}_pred_category'], labels=cats)
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(cm / cm.sum(1, keepdims=True), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(cats)), cats, rotation=90, fontsize=7); ax.set_yticks(range(len(cats)), cats, fontsize=7)
for i in range(len(cats)):
    for j in range(len(cats)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                             color='white' if cm[i, j] > cm[i].sum() / 2 else 'black')
ax.set_xlabel('Predicted category'); ax.set_ylabel('True category')
ax.set_title(f'Test confusion matrix – category ({DEPLOY} + SVM)')
plt.tight_layout(); plt.savefig(RESULTS / 'category_confusion_matrix.png', bbox_inches='tight'); plt.show()

## 10. Run manifest and results highlights

In [ ]:
def ver(p):
    try: return metadata.version(p)
    except Exception: return None

manifest = {
    'task': 'ticket subcategory (59) -> category (17) classification',
    'ENCODER_BACKEND': 'sentence-transformers' if HAS_MINILM else 'MiniLM SKIPPED (TF-IDF baseline only)',
    'encoder_message': encoder_message,
    'encoders_run': list(ENCODERS), 'deployed_encoder': DEPLOY,
    'selected_hyperparameters': BEST, 'auto_route_thresholds': thresholds, 'temperatures': temperatures,
    'target_auto_route_accuracy': TARGET_AUTO_ACCURACY, 'seed': SEED,
    'selection': 'C and class_weight by validation subcategory macro-F1; encoder by unseen-wording CV on train+validation',
    'versions': {p: ver(p) for p in ['sentence-transformers', 'transformers', 'torch', 'scikit-learn', 'numpy', 'pandas', 'matplotlib', 'joblib']},
    'python': platform.python_version(),
    'limitations': ['synthetic data', 'heavily templated text (200 short-description templates)',
                    'labels assumed correct', 'MiniLM-L6 is English-only; Japanese bodies rely on the English short description'],
}
(RESULTS / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))

T = test_metrics.set_index(['model', 'split'])
print('=' * 78)
print('ENCODER BACKEND:', manifest['ENCODER_BACKEND'])
print('=' * 78)
for e in ENCODERS:
    r = T.loc[(f'{e}_svm', 'test')]
    print(f"{e:>6} + SVM | test subcat acc {r.subcat_accuracy:.3f}  macro-F1 {r.subcat_macro_f1:.3f}  "
          f"top-3 {r.subcat_top3_accuracy:.3f} | category acc {r.category_accuracy:.3f} | "
          f"auto-route {r.auto_route_coverage:.1%} at {r.auto_route_accuracy:.1%} acc")
r = T.loc[('template_lookup', 'test')]
print(f"template lookup | test subcat acc {r.subcat_accuracy:.3f}  macro-F1 {r.subcat_macro_f1:.3f}")
print('-' * 78)
print('Unseen wording (grouped 5-fold CV on train+validation), subcategory macro-F1:')
print(unseen.pivot_table(index='model', columns='split', values='subcat_macro_f1').round(3).to_string())
print('-' * 78)
print(f'Deployed: {DEPLOY} + SVM  {BEST[DEPLOY]}  threshold {thresholds[DEPLOY]:.2f}')
print(f'Unlabelled tickets auto-routed: {sug.auto_route.sum()} / {len(sug)}')